## 数据读取与可视化  
此处是读取微震原数据，后续将对该数据进行一系列模态分解

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('../data/raw/8_15-9_25.csv')

# 划分测试集、验证集
l_1 = int(len(df['quake_level']) * 0.7)
l_2 = int(len(df['quake_level']) * 0.85)

quake_level_train = df['quake_level'][:l_1].to_numpy()
quake_level_val = df['quake_level'][l_1:l_2].to_numpy()
quake_level_test = df['quake_level'][l_2:].to_numpy()
quake_level_array = np.concatenate((quake_level_train, quake_level_val, quake_level_test), axis=0)
t = np.arange(1, len(quake_level_array)+1)

# t1、t2、t3作为可视化横坐标
t1 = np.arange(1, len(quake_level_train)+1)
t2 = np.arange(len(quake_level_train)+1, len(quake_level_train)+len(quake_level_val)+1)
t3 = np.arange(len(quake_level_train)+len(quake_level_val)+1, len(quake_level_train)+len(quake_level_val)+len(quake_level_test)+1)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimSun']
plt.figure(figsize=(12, 2))
plt.plot(t1, quake_level_train, 'b', label='train_data')
plt.plot(t2, quake_level_val, 'g', label='val_data')
plt.plot(t3, quake_level_test, 'r', label='test_data')
plt.title("Original Signal")
plt.ylabel("Amplitude")
plt.legend(bbox_to_anchor=(0.8, 1))
plt.tight_layout()
plt.show()

## CEEMDAN分解

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PyEMD import CEEMDAN

# CEEMDAN分解过程
def ceemdan_decomposition(signal):
    # 实例化CEEMDAN对象
    ceemdan = CEEMDAN()
    
    # 设置噪声参数
    ceemdan.noise_seed(12345)  # 固定噪声种子，使每次分解结果一致
    
    # 对信号进行CEEMDAN分解
    imfs = ceemdan(signal)
    
    return imfs
    
# plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['font.sans-serif'] = ['SimSun']
plt.rcParams['axes.unicode_minus'] = False 

# 可视化结果
def plot_imfs(t, signal, imfs):
    num_imfs = imfs.shape[0]
    
    # 原始信号
    plt.figure(figsize=(16, 18))
    plt.subplot(num_imfs + 1, 1, 1)
    plt.plot(t, signal, 'black')  # 原始信号用红色
    plt.title("原始数据")
    plt.ylabel("天")
    plt.ylabel("能量强度")

    # 使用 cmap 生成颜色
    cmap = plt.get_cmap('tab10')  # 选择一个颜色映射，如 'viridis', 'plasma', 'inferno', 'magma', 'cividis'
    cmap = plt.get_cmap('viridis')
    colors = [cmap(i / num_imfs) for i in range(num_imfs)]

    # 绘制每个IMF
    for i in range(num_imfs):
        plt.subplot(num_imfs + 1, 1, i + 2)
        plt.plot(t, imfs[i], color=colors[i])  # 使用颜色列表中的颜色
        plt.title(f"C-IMF {i + 1}")
        # plt.ylabel("Amplitude")
    
    plt.tight_layout()
    plt.savefig('sine_wave.svg', format='svg', dpi=300)
    plt.show()

# 进行CEEMDAN分解
imfs = ceemdan_decomposition(quake_level_array)
#sample_imfs = ceemdan_decomposition(sample_data)

# 可视化结果
plot_imfs(t, quake_level_array, imfs)

## 样本熵的计算

In [ ]:
import numpy as np
import nolds

def max_distance(x_i, x_j):
    """
    计算两个向量之间的最大距离
    """
    return np.max(np.abs(x_i - x_j))

def construct_vectors(time_series, m):
    """
    构建长度为 m 的子序列
    """
    N = len(time_series)
    vectors = np.array([time_series[i:i + m] for i in range(N - m + 1)])
    return vectors

def count_matches(vectors, r):
    """
    计算满足条件的匹配数
    """
    N = len(vectors)
    count = 0
    for i in range(N):
        for j in range(i + 1, N):
            if max_distance(vectors[i], vectors[j]) < r:
                count += 1
    return count

def sample_entropy(time_series, m, r):
    """
    计算时间序列的样本熵

    参数：
    time_series -- 时间序列 (list 或 numpy array)
    m -- 嵌入维度 (int)
    r -- 公差 (tolerance, float)

    返回值：
    样本熵 (float)
    """
    N = len(time_series)
    
    # 构建 m 和 m+1 的向量
    Xm = construct_vectors(time_series, m)
    Xm1 = construct_vectors(time_series, m + 1)
    
    # 计算 B^m(r) 和 A^m(r)
    B_m_r = count_matches(Xm, r)
    A_m_r = count_matches(Xm1, r)
    
    # 如果 B_m_r 为0，样本熵无法计算，返回无穷大
    if B_m_r == 0:
        return np.inf
    
    return -np.log(A_m_r / B_m_r)

# 自定义样本熵函数计算(速度较慢)
# for i, imf in enumerate(sample_imfs):
#     sampen = sample_entropy(imf, 2, 0.2 * np.std(imf))
#     print(f"imf{i+1}样本熵: {sampen}")

print("===="*10)
# 调用nolds计算样本熵
for i, imf in enumerate(imfs):
    sampen = nolds.sampen(imf, emb_dim=1, tolerance=0.1 * np.std(imf))
    print(f"imf{i+1}样本熵: {sampen}")

### 实验——m、r参数的变化是否影响样本熵对高低频序列的判别  
由于要使用样本熵来区分高频、低频、趋势序列，故在这里验证，不同的参数m、r对计算样本熵以区分高低频序列无太大影响

In [60]:
# 计算所有infs样本熵
def sampen_list(m, r, imfs):
    sampen_list = []
    for i, imf in enumerate(imfs):
        sampen = nolds.sampen(imf, emb_dim=m, tolerance=r * np.std(imf))
        sampen_list.append(sampen)
    return sampen_list

# 不同参数m、r
sampen_list1 = sampen_list(1, 0.1, imfs)
sampen_list2 = sampen_list(2, 0.1, imfs)
sampen_list3 = sampen_list(1, 0.2, imfs)
sampen_list4 = sampen_list(2, 0.2, imfs)
x = list(range(1, len(sampen_list1)+1))

In [ ]:
# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimSun']
plt.rcParams['font.size'] = 12 
plt.rcParams['axes.unicode_minus'] = False 
cmap = plt.get_cmap('viridis')
colors = [cmap(i / 5) for i in range(5)]
plt.figure(figsize=(8, 6))
plt.plot(x, sampen_list1,  marker='o', label="m=1,r=0.1", linestyle='-.',color=colors[0])
plt.plot(x, sampen_list3,  marker='^', label="m=1,r=0.2", linestyle='-.',color=colors[1])
plt.plot(x, sampen_list2,  marker='s', label="m=2,r=0.1", linestyle='-',color=colors[2])
plt.plot(x, sampen_list4,  marker='*', label="m=2,r=0.2", linestyle=':',color=colors[3])
plt.legend(loc='upper right')
plt.xlabel("C-IMFs")
plt.ylabel("样本熵")
# plt.title("不同序列样本熵随着m、r参数变化图")
plt.tight_layout()
plt.grid(True)
plt.savefig('样本熵.svg', format='svg', dpi=300, bbox_inches='tight')
plt.show()

## 重构序列  
上述根据CEEMDAN方法将原始序列分解为多个C-IMF序列，并对每个IMF计算其样本熵。这里根据样本熵的大小将C-IMFs序列重构为高频、低频和趋势三个序列。

In [ ]:
high_frequency = np.sum(imfs[:4], axis=0)
trend_sequence = np.sum(imfs[4:7], axis=0)
low_frequency = np.sum(imfs[-3:], axis=0)
# 选择颜色映射
cmap = plt.get_cmap('Accent')  # 选择颜色映射
cmap = plt.get_cmap('viridis')
# 生成颜色列表
num_colors = 3  # 假设你有5个类别
colors = [cmap(i / num_colors) for i in range(num_colors)]
plt.figure(figsize=(12, 4.5))
plt.subplot(3, 1, 1)
plt.plot(high_frequency,  color=colors[0])
plt.title("高频序列")
plt.subplot(3, 1, 2)
plt.plot(trend_sequence,  color=colors[1])
plt.title("中频序列")
plt.subplot(3, 1, 3)
plt.plot(low_frequency,  color=colors[2])
plt.title("低频序列")
plt.tight_layout()
plt.savefig('重构序列.svg', format='svg', dpi=300, bbox_inches='tight')
plt.show()

## VMD分解  
对上述高频、低频序列进行VMD分解，趋势序列不变

In [9]:
# VMD分解
from vmdpy import VMD
alpha = 7000
tau = 0.
K = 8
DC = 0
init = 1
tol = 1e-7  

"""  
alpha、tau、K、DC、init、tol 六个输入参数的无严格要求； 
alpha 带宽限制 经验取值为 抽样点长度 1.5-2.0 倍； 
tau 噪声容限 ；
K 分解模态（IMF）个数； 
DC 合成信号若无常量，取值为 0；若含常量，则其取值为 1； 
init 初始化 w 值，当初始化为 1 时，均匀分布产生的随机数； 
tol 控制误差大小常量，决定精度与迭代次数
"""
# 将高频序列分解为10个imf，将低频序列分解为个imf
u, u_hat, omega = VMD(high_frequency , alpha, tau, 12, DC, init, tol)  
u1, u_hat1, omega1 = VMD(low_frequency , alpha, tau, 6, DC, init, tol) 
u2, u_hat2, omega2 = VMD(trend_sequence , alpha, tau, 6, DC, init, tol) 

In [10]:
u2.shape

(6, 2352)

In [ ]:
import math
# 可视化vmd分解结果
#l = u.shape[0] + u1.shape[0]
plt.figure(figsize=(12, 20))
plt.rcParams['font.sans-serif'] = ['SimSun']
plt.rcParams['font.size'] = 10
cmap = plt.get_cmap('viridis')  # 使用 'viridis' 颜色映射

for i in range(u.shape[0]):
    colors = [cmap(i / u.shape[0]) for i in range(u.shape[0])]
    plt.subplot(16, 1, i + 1)
    plt.plot(np.arange(1, u.shape[1]+1), u[i, :], color=colors[i])
    plt.title(f'V-IMF {i + 1}')
plt.tight_layout()
plt.savefig('vmd分解.svg', format='svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import math
# 可视化vmd分解结果
l = u.shape[0] + u1.shape[0]
plt.figure(figsize=(16, 16))
for i in range(u.shape[0]):
    plt.subplot(int(math.ceil(l/2.0)), 2, i + 1)
    plt.plot(np.arange(1, u.shape[1]+1), u[i, :])
    plt.title(f'V-IMF {i + 1}')
# for i in range(u1.shape[0]):
#     plt.subplot(int(math.ceil(l/2.0)), 2, i + u.shape[0] + 1)
#     plt.plot(np.arange(1, u1.shape[1]+1), u1[i, :])
#     plt.title(f'V-IMF {i + 11}')
plt.tight_layout()
plt.savefig('vmd分解.svg', format='svg', dpi=300)
plt.show()